# Live exercise: the cleaning as two functions, worked solution

The same code lives in `sessions/session-09/solutions/cleaning.py`, which
is the version session 10 picks up. That file has one extra step, which
makes `clean_plays` work even on a file read without `keep_default_na=False`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy" / "plays_messy.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

MESSY = "data/messy/plays_messy.csv"
CLEAN = "data/clean/plays.csv"

clean = pd.read_csv(CLEAN)

## 1. `load_messy`

One line. The value of the function is the name and the docstring: anyone
reading `load_messy(path)` knows what comes back, and the decision about
`keep_default_na` is written down in one place.

In [ ]:
def load_messy(path):
    """Return the messy CSV at path as a DataFrame, with nothing guessed.

    keep_default_na=False stops pandas turning "NA" and empty cells into
    missing values on its own. Every text column arrives exactly as it was
    written, and clean_plays decides what counts as missing.
    """
    return pd.read_csv(path, keep_default_na=False)


raw = load_messy(MESSY)
assert raw.shape == (2223, 9)
assert (raw["country"] == "NA").sum() == 49

## 2. The two helpers

Each one is a single idea from notebook 01, with a name.

In [ ]:
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"]


def parse_dates(text):
    """Return a column of dates written in any of DATE_FORMATS as datetimes.

    Each format is tried on purpose, one at a time. A single blanket parse
    reads 05/01/2024 as the 1st of May, without a word of warning.
    """
    dates = pd.to_datetime(text, format=DATE_FORMATS[0], errors="coerce")
    for date_format in DATE_FORMATS[1:]:
        dates = dates.fillna(pd.to_datetime(text, format=date_format, errors="coerce"))
    assert dates.notna().all(), "some dates are in none of DATE_FORMATS"
    return dates


def fill_from_artist(df, column):
    """Return column with its gaps filled from the same artist's other rows."""
    lookup = df.dropna(subset=[column]).groupby("artist_name")[column].first()
    return df[column].fillna(df["artist_name"].map(lookup))


test = pd.Series(["2024-08-03", "03/08/2024", "03-08-2024"])
assert parse_dates(test).nunique() == 1        # three spellings, one date
print(parse_dates(test))

A three-row test on dates you can check in your head is worth more than a
glance at 2,223 parsed rows. All three are the 3rd of August.

The `assert` inside `parse_dates` matters as much as the parsing. If next
month's export adds a fourth format, the function stops and says so,
instead of quietly handing back `NaT`.

## 3. `clean_plays`

In [ ]:
COLUMNS = list(clean.columns)


def clean_plays(df):
    """Return a cleaned copy of the messy plays, shaped like data/clean/plays.csv.

    Duplicates go, text is made consistent, dates are parsed deliberately,
    minutes become numbers, gaps in genre and country are filled from the
    artist's other plays, and a missing device becomes "unknown". Missing
    minutes stay NaN. The input DataFrame is not changed.
    """
    df = df.rename(columns={"minutes played ": "minutes_played"})
    df = df.drop_duplicates().copy()
    assert df["play_id"].is_unique, "a play_id appears twice with different values"

    df["genre"] = df["genre"].str.strip().str.title().replace("", pd.NA)
    df["device"] = df["device"].str.strip().str.lower().replace("", "unknown")
    df["country"] = df["country"].str.strip().replace(["NA", ""], pd.NA)
    df["genre"] = fill_from_artist(df, "genre")
    df["country"] = fill_from_artist(df, "country")

    df["played_at"] = parse_dates(df["played_at"]).dt.strftime("%Y-%m-%d")

    minutes = df["minutes_played"].str.replace("min", "").str.replace(" ", "")
    df["minutes_played"] = pd.to_numeric(minutes, errors="coerce")
    unreadable = df["minutes_played"].isna() & (minutes != "")
    assert not unreadable.any(), "some minutes could not be read as numbers"

    return df[COLUMNS].sort_values("play_id").reset_index(drop=True)

Some choices worth a sentence each:

* **Duplicates go first.** They are exact copies, so it makes no difference
  to the result, and every later step has 40 fewer rows to handle. The
  `assert` straight after checks the stronger promise, that each play is now
  there once.
* **`df = df.drop_duplicates().copy()`.** The `.copy()` makes it obvious
  that this is a new DataFrame, so changing its columns can never reach
  back into `raw`. That is how the function keeps its promise not to change
  its input.
* **The order of the text steps.** Blanks are turned into missing values
  before `fill_from_artist`, because `fillna` cannot see an empty string.
* **The minutes check.** After `coerce`, the only missing values should be
  the cells that were blank. If anything else turned into `NaN`, the
  `assert` stops the function. In notebook 01 that was a cell you had to
  remember to run; here it runs every time.

This version also removes spaces from minutes. The export writes a
thousands separator as a space, like `1 234.50`. No play in this file is
long enough to have one, so it changes nothing today, but nothing else
would catch it if one turned up.

## 4. Check it

In [ ]:
plays = clean_plays(raw)

assert len(plays) == 2183
assert list(plays.columns) == list(clean.columns)
assert (plays.dtypes == clean.dtypes).all()
assert plays["play_id"].equals(clean["play_id"])
assert raw.shape == (2223, 9)                 # the input was not changed

plays.head()

Every promise holds. Now, column by column:

In [ ]:
for column in COLUMNS:
    same = (plays[column] == clean[column]).sum()
    print(f"{column:15} {same:>5} of {len(clean)}")

Seven columns match completely. `minutes_played` differs in 94 cells, the
blanks, and `device` in 69, the plays now labelled `"unknown"`. Both are
the information the file had already lost.

If your numbers came out differently, the usual suspects:

* `device` differs in 240 rows: a missing `.str.strip()`
* `played_at` differs in 158 rows: `format="mixed"` crept back in (159 in
  notebook 01, one of which was a duplicate)
* `minutes_played` differs in 187 rows: `" min"` was not removed before
  converting
* the dtypes check fails on `played_at`: the dates were left as datetimes
  instead of being turned back into text

In [ ]:
assert (plays["minutes_played"] == clean["minutes_played"]).sum() == 2183 - 94
assert (plays["device"] == "unknown").sum() == 69

## If you have time: the chart, from your own data

In [ ]:
by_genre = plays.groupby("genre")["minutes_played"].mean().sort_values(ascending=False)
top_genre = by_genre.index[0]

os.makedirs("output", exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(by_genre.index, by_genre.values, color="#4f6ddb")
ax.set_title(f"{top_genre} plays last longest: {by_genre.iloc[0]:.1f} minutes on average")
ax.set_ylabel("average minutes per play")
ax.bar_label(bars, fmt="{:.1f}")
ax.spines[["top", "right"]].set_visible(False)

fig.savefig("output/minutes_by_genre_cleaned.png", dpi=150, bbox_inches="tight")
plt.show()

Same answer, Jazz at 6.6 minutes, and the same order of genres. Put the
two sets of labels side by side, though:

In [ ]:
from_clean = clean.groupby("genre")["minutes_played"].mean()

comparison = pd.DataFrame({
    "clean file": from_clean.map("{:.1f}".format),
    "our cleaning": by_genre.map("{:.1f}".format),
})
comparison[comparison["clean file"] != comparison["our cleaning"]]

Three labels move by 0.1. The averages leave out the 94 plays with no
minutes, and those plays were not spread evenly across genres, so each
genre's average shifts a little. Small enough not to change the story,
big enough to show that even "leave them missing" is a decision with a
visible effect.

In [ ]:
assert top_genre == "Jazz"
assert len(comparison[comparison["clean file"] != comparison["our cleaning"]]) == 3